# Single-stream masked JEPA prototype

This notebook is a **small Health&Gait feasibility experiment**. It tests whether a single video stream can learn non-collapsed representations by predicting masked target-encoder features. It intentionally follows the masked V-JEPA training pattern: the context and target encoders see complementary token regions from the same clip.

It is **not** the final CoDy-JEPA model and does not implement counterfactual token swapping, factorized streams, or explicit future-dynamics prediction. Success requires more than falling loss: validation representations must retain variance/effective rank, and shuffling video context must make prediction worse.

`Run All` is safe: it validates data and runs a tiny CPU smoke test, but the full CUDA job remains behind `RUN_FULL_TRAINING = False`.

In [ ]:
from pathlib import Path
import json
import math
import random
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
from torch.utils.data import DataLoader

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / 'src').exists() else NOTEBOOK_CWD.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from cody_jepa.data import (
    HealthGaitLoaderConfig,
    audit_healthgait_clip_quality,
    build_healthgait_datasets_from_config,
    build_healthgait_loaders_from_config,
    find_repo_root,
    healthgait_manifest_path,
)
from cody_jepa.single_stream_jepa import (
    DEFAULT_MASK_GROUPS,
    MODEL_ARCHITECTURE,
    build_models,
    load_checkpoint,
    multiblock_mask,
    train_jepa,
    video_from_batch,
)

RUN_FULL_TRAINING = True

CONFIG = {
    'seed': 0,
    # batch_size is a physical microbatch; four microbatches make an effective 64.
    'batch_size': 16,
    'accumulation_steps': 4,
    'steps': 3_900,
    'num_epochs': 100,
    'lr': 1e-4,
    'start_lr': 1e-5,
    'warmup_steps': 200,
    'min_lr': 1e-6,
    'weight_decay': 0.04,
    'grad_clip': 1.0,
    'ema_start': 0.998,
    'ema_end': 1.0,
    'loss_exp': 1.0,

    'num_frames': 16,
    'img_size': 112,
    'patch_size': 8,
    'tubelet_size': 2,
    'in_channels': 1,
    'num_tokens': 1568,
    'min_context_tokens': 64,
    'input_mean': 0.5,
    'input_std': 0.5,

    'embed_dim': 384,
    'hidden_dim': 1536,
    'num_heads': 6,
    'num_layers': 6,
    'pred_dim': 192,
    'pred_depth': 6,
    'dropout': 0.0,
    'uniform_power': True,
    'norm_eps': 1e-6,

    'amp_dtype': 'bfloat16',
    'compile': False,  # enable only after the eager CUDA preflight succeeds
    'tf32': True,
    'required_device': 'cuda',
    'eval_every_epochs': 5,
    'train_eval_every_epochs': 0,
    'checkpoint_every_epochs': 1,
    'shortcut_diagnostic_batches': 1,
    'selection_metric': 'subject_balanced_loss',
    'min_feature_std': 1e-3,
    'max_near_zero_variance_fraction': 0.5,
    'min_effective_rank_ratio': 0.05,
    'min_context_shuffle_loss_gap': 1e-3,
}

assert CONFIG['num_tokens'] == (
    CONFIG['num_frames'] // CONFIG['tubelet_size']
    * (CONFIG['img_size'] // CONFIG['patch_size']) ** 2
)
print(f'architecture: {MODEL_ARCHITECTURE}')
print(f'effective batch: {CONFIG["batch_size"] * CONFIG["accumulation_steps"]}')

In [ ]:
REPO_ROOT = find_repo_root()
MANIFEST_CSV = healthgait_manifest_path(REPO_ROOT)
LOADER_CONFIG = HealthGaitLoaderConfig(
    manifest_csv=MANIFEST_CSV,
    repo_root=REPO_ROOT,
    split='train',
    clip_length=CONFIG['num_frames'],
    image_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['seed'],
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    prefetch_factor=1,
    train_crop_scale=(0.90, 1.0),
    train_horizontal_flip_prob=0.5,
    strict_frame_sequence=True,
    image_verify_mode='all' if RUN_FULL_TRAINING else 'sample',
    inventory_hash_mode='full' if RUN_FULL_TRAINING else 'sample',
    allowed_data_root=REPO_ROOT / 'data' / 'healthgait',
    eval_windows=3,
    drop_last_train=True,
)

# Build each dataset once; the validation dataset reuses the validated manifest inventory.
train_ds, val_ds = build_healthgait_datasets_from_config(LOADER_CONFIG)
train_loader, val_loader = build_healthgait_loaders_from_config(
    LOADER_CONFIG, datasets=(train_ds, val_ds)
)
portable_loader_config = {
    key: value
    for key, value in LOADER_CONFIG.as_dict().items()
    if key not in {'manifest_csv', 'repo_root', 'allowed_data_root'}
}
DATA_CONTRACT = {
    'loader_config': portable_loader_config,
    'train_dataset': train_ds.signature(),
    'val_dataset': val_ds.signature(),
}
print(json.dumps({
    'train_sequences': len(train_ds.samples),
    'train_clips_per_epoch': len(train_ds),
    'val_sequences': len(val_ds.samples),
    'val_deterministic_windows': len(val_ds),
    'data_contract': DATA_CONTRACT,
}, indent=2, sort_keys=True))

In [ ]:
# Adversarial data preflight: decode one deterministic clip from every train and
# validation sequence, then separately check the tensor/metadata loss boundary.
quality_summary = audit_healthgait_clip_quality((train_ds, val_ds))
preflight_loader = DataLoader(train_ds, batch_size=4, shuffle=False, num_workers=0)
preflight_batch = next(iter(preflight_loader))
preflight_video = video_from_batch(
    preflight_batch, torch.device('cpu'), CONFIG, expected_split='train'
)
train_subjects = {sample['subject_id'].casefold() for sample in train_ds.samples}
val_subjects = {sample['subject_id'].casefold() for sample in val_ds.samples}
assert train_subjects.isdisjoint(val_subjects)
assert torch.isfinite(preflight_video).all()
print({
    'batch_shape': tuple(preflight_batch['video'].shape),
    'pixel_range': (
        float(preflight_batch['video'].min()),
        float(preflight_batch['video'].max()),
    ),
    'quality_summary': quality_summary,
    'train_subjects': len(train_subjects),
    'val_subjects': len(val_subjects),
})

In [ ]:
def smoke_test_training_loop():
    smoke_cfg = {
        **CONFIG,
        'batch_size': 2, 'accumulation_steps': 1, 'steps': 1, 'num_epochs': 1,
        'lr': 1e-3, 'start_lr': 1e-3, 'warmup_steps': 1, 'min_lr': 1e-5,
        'ema_start': 0.9, 'ema_end': 0.9,
        'num_frames': 4, 'img_size': 16, 'patch_size': 4, 'tubelet_size': 2,
        'num_tokens': 32, 'min_context_tokens': 4,
        'embed_dim': 12, 'hidden_dim': 24, 'num_heads': 3, 'num_layers': 1,
        'pred_dim': 12, 'pred_depth': 1,
        'amp_dtype': None, 'compile': False, 'tf32': False,
        'required_device': 'cpu', 'eval_every_epochs': 1,
    }
    generator = torch.Generator().manual_seed(smoke_cfg['seed'])
    train_samples = [{
        'video': torch.rand(4, 1, 16, 16, generator=generator),
        'split': 'train', 'sequence_id': f'train-{index}', 'subject_id': f'T{index}',
    } for index in range(2)]
    val_samples = [{
        'video': torch.rand(4, 1, 16, 16, generator=generator),
        'split': 'val', 'sequence_id': f'val-{index}', 'subject_id': f'V{index}',
    } for index in range(2)]
    smoke_train = DataLoader(
        train_samples, batch_size=2, shuffle=True,
        generator=torch.Generator().manual_seed(smoke_cfg['seed']),
    )
    smoke_val = DataLoader(val_samples, batch_size=2, shuffle=False)
    result = train_jepa(
        smoke_cfg, smoke_train, smoke_val, {'dataset': 'synthetic-smoke-v1'},
        checkpoint_dir=None, device='cpu',
    )
    assert result['global_step'] == 1
    assert result['completed_epochs'] == 1
    assert result['history'][0]['val']['effective_rank'] > 0
    assert not any(p.requires_grad for p in result['target_encoder'].parameters())
    assert all(p.grad is None for p in result['target_encoder'].parameters())
    masks_a = multiblock_mask(smoke_cfg, 2, random.Random(99))
    masks_b = multiblock_mask(smoke_cfg, 2, random.Random(99))
    assert all(torch.equal(a['ctx'], b['ctx']) for a, b in zip(masks_a, masks_b))
    return result['history'][0]

smoke_metrics = smoke_test_training_loop()
smoke_metrics

In [ ]:
# CPU-only structural preflight. The full CUDA run still performs the real BF16
# forward/backward path and fails fast if the selected GPU is unsuitable.
production_masks = multiblock_mask(CONFIG, 2, random.Random(CONFIG['seed']))
for group in production_masks:
    assert group['ctx'].shape[0] == 2 and group['pred'].shape[0] == 2
    assert not torch.isin(group['ctx'][0], group['pred'][0]).any()
production_context, production_target, production_predictor = build_models(
    CONFIG, torch.device('cpu')
)
parameter_counts = {
    'context_encoder': sum(p.numel() for p in production_context.parameters()),
    'target_encoder': sum(p.numel() for p in production_target.parameters()),
    'predictor': sum(p.numel() for p in production_predictor.parameters()),
}
assert production_context.num_patches == CONFIG['num_tokens']
del production_context, production_target, production_predictor
print({
    'parameter_counts': parameter_counts,
    'mask_shapes': [(g['label'], tuple(g['ctx'].shape), tuple(g['pred'].shape)) for g in production_masks],
    'input_microbatch_mib': (
        CONFIG['batch_size'] * CONFIG['num_frames'] * CONFIG['img_size'] ** 2 * 4 / 2**20
    ),
})

In [ ]:
RESUME_CHECKPOINT = None  # e.g. OUTPUT_DIR / 'latest.pt'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'single-stream-jepa-healthgait-v3'

result = None
if RUN_FULL_TRAINING:
    if LOADER_CONFIG.image_verify_mode != 'all' or LOADER_CONFIG.inventory_hash_mode != 'full':
        raise RuntimeError('full training requires full frame decode verification and hashing')
    if not torch.cuda.is_available():
        raise RuntimeError('full training requires an allocated CUDA GPU')
    if RESUME_CHECKPOINT is None and (OUTPUT_DIR / 'latest.pt').exists():
        raise FileExistsError(
            f'{OUTPUT_DIR / "latest.pt"} already exists; choose a new OUTPUT_DIR or resume it'
        )
    resume_state = (
        load_checkpoint(RESUME_CHECKPOINT) if RESUME_CHECKPOINT is not None else None
    )
    result = train_jepa(
        CONFIG,
        train_loader,
        val_loader,
        DATA_CONTRACT,
        checkpoint_dir=OUTPUT_DIR,
        resume_state=resume_state,
    )
    print({
        'latest': str(OUTPUT_DIR / 'latest.pt'),
        'best_loss': str(OUTPUT_DIR / 'best_loss.pt'),
        'best_healthy': str(OUTPUT_DIR / 'best_healthy.pt'),
        'best_epoch': result['best_epoch'],
        'best_val_loss': result['best_val_loss'],
        'best_healthy_epoch': result['best_healthy_epoch'],
        'termination_reason': result['termination_reason'],
    })
else:
    print('Full training skipped. Set RUN_FULL_TRAINING=True on a CUDA worker after reviewing CONFIG.')

In [ ]:
if result is None:
    print('No full-run result to plot.')
else:
    history = result['history']
    steps = [row['step'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    train_cosine = [row['train_cosine'] for row in history]
    eval_rows = [row for row in history if row['val'] is not None]
    eval_steps = [row['step'] for row in eval_rows]
    val_loss = [row['val']['loss'] for row in eval_rows]
    val_cosine = [row['val']['cosine'] for row in eval_rows]
    effective_rank = [row['val']['effective_rank'] for row in eval_rows]
    feature_std = [row['val']['feature_std'] for row in eval_rows]
    near_zero_fraction = [row['val']['near_zero_variance_fraction'] for row in eval_rows]
    shuffle_gap = [row['val']['context_shuffle_loss_gap'] for row in eval_rows]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    axes[0, 0].plot(steps, train_loss, label='train')
    axes[0, 0].plot(eval_steps, val_loss, 'o-', label='validation')
    axes[0, 0].set_title('JEPA loss')
    axes[0, 1].plot(steps, train_cosine, label='train')
    axes[0, 1].plot(eval_steps, val_cosine, 'o-', label='validation')
    axes[0, 1].set_title('Cosine similarity')
    axes[1, 0].plot(eval_steps, effective_rank, 'o-')
    axes[1, 0].set_title('Validation effective rank')
    axes[0, 2].plot(eval_steps, feature_std, 'o-')
    axes[0, 2].set_title('Validation feature std')
    axes[1, 1].plot(eval_steps, near_zero_fraction, 'o-')
    axes[1, 1].set_title('Near-zero variance fraction')
    axes[1, 2].plot(eval_steps, shuffle_gap, 'o-')
    axes[1, 2].axhline(0, color='black', linewidth=1)
    axes[1, 2].set_title('Wrong-subject context loss gap')
    for axis in axes.flat:
        axis.set_xlabel('optimizer step')
        axis.xaxis.set_major_locator(MaxNLocator(8, integer=True))
        axis.grid(True, alpha=0.3)
        if axis.get_legend_handles_labels()[0]:
            axis.legend(frameon=False)
    plt.show()